In [ ]:
### backtestingvisualization.ipynb is a modular and agnostic system that visualizes backtesting
### data for parameterization and unsupervised learning of values.

In [ ]:
## Import dependencies
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Set style, deprecated as of 09-07-2025 need to be tuned
sns.set(context="notebook", style="whitegrid")


In [ ]:
## --- Parameters ---
INPUT_CSV_PATH = 'verificationhistoricaldata/M1dataflow.csv'  # e.g. '/path/to/full_labeled_USD_JPY.csv'
OUTPUT_PNG_PATH = 'verificationhistoricaldocumentation/Visualizations/0125to0825M1.png'

# Column names
COL_TSEC = 'T'
COL_PRICE = 'value'
COL_VOLUME = 'volume'
COL_BUY = 'B'
COL_SELL = 'S'
COL_MARGIN = 'Margin'
COL_THETA = 'Theta'
COL_MU = 'Mu'
COL_ACTIONHAT = 'ActionHat'
COL_TRUEACTION = 'TrueAction'

## TrueAction Order
TRUEACTION_ORDER = [0, 1, 2]


In [ ]:
required_cols = [
    COL_TSEC, COL_PRICE, COL_VOLUME, COL_BUY, COL_SELL,
    COL_MARGIN, COL_THETA, COL_MU, COL_ACTIONHAT, COL_TRUEACTION
]

try:
    df  # See if df already exists
    _has_df = True
except NameError:
    _has_df = False

if not _has_df:
    if not INPUT_CSV_PATH:
        raise ValueError("No DataFrame named 'df' in memory and INPUT_CSV_PATH is empty. "
                         "Set INPUT_CSV_PATH or define 'df' before running.")
    df = pd.read_csv(INPUT_CSV_PATH)

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"DataFrame is missing required columns: {missing}")

df = df.copy()
display(df.head(3))
print(f"Rows: {len(df):,}")


,t_sec,value,volume,Buy,Sell,margin,theta,mu,ActionHat,TrueAction
0,0,147.158,245,-0.0314,-0.0314,-0.0314,0.0,-0.0314,2,2
1,60,147.167,219,-0.0314,-0.0314,-0.0314,0.0,-0.0314,2,2
2,120,147.160,156,-0.0314,-0.0314,-0.0314,0.0,-0.0314,2,2


Rows: 6,973


In [ ]:
def plot_bs_scatter(
    data: pd.DataFrame,
    x_col: str = COL_SELL,  # x = S
    y_col: str = COL_BUY,   # y = B
    hue_col: str = 'green',
    output_path: str = OUTPUT_PNG_PATH,
    title: str = "B vs S",
    point_size: int = 14,
    alpha: float = 0.7
):
    """plot_bs_scatter makes a Seaborn plot of BxS observed manifold. It outputs and saves png files
    as well as prints a small message.
    
    Inputs:
    data: pd.DataFrame  - the data
    x_col: str = COL_SELL - the sell information
    y_col: str = COL_BUY - the buy information
    hue_col: str = 'green' - the color of the dots
    output_path: str = OUTPUT_PNG_PATH - the output path
    title: str = "B vs S" - the title of the scatterplot
    point_size: int = 14 - the point size
    alpha: float = 0.7 - the alpha value 
    """

    fig, ax = plt.subplots(figsize=(7, 7))
    sns.scatterplot(
        data=data,
        x=x_col, y=y_col,
        hue=hue_col,
        s=point_size, alpha=alpha,
        linewidth=0, ax=ax
    )

    # Diagonal (B = S)
    min_lim = min(data[x_col].min(), data[y_col].min())
    max_lim = max(data[x_col].max(), data[y_col].max())
    ax.plot([min_lim, max_lim], [min_lim, max_lim], linestyle='--', linewidth=1, color='black')

    ax.set_xlabel('S (Sell)')
    ax.set_ylabel('B (Buy)')
    ax.set_title(title)
    ax.set_xlim(min_lim, max_lim)
    ax.set_ylim(min_lim, max_lim)
    ax.set_aspect('equal', adjustable='box')

    ax.legend(title=hue_col, bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
    plt.tight_layout()

    fig.savefig(output_path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved scatter to: {output_path}")


In [ ]:
def plot_uv_scatter(
    data: pd.DataFrame,
    x_col: str = COL_THETA,  # x = mu
    y_col: str = COL_MU,   # y = theta
    hue_col: str = 'blue',
    output_path: str = OUTPUT_PNG_PATH,
    title: str = "Mu vs Theta — colored by TrueAction (0/1/2)",
    point_size: int = 14,
    alpha: float = 0.7
):
    """plot_uv_scatter makes a Seaborn plot of UxV observed manifold. It outputs and saves png files
    as well as prints a small message.
    
    Inputs:
    data: pd.DataFrame - the data
    x_col: str = COL_THETA - the sell information
    y_col: str = COL_MU - the color of the dots
    hue_col: str = 'blue' - the output path
    output_path: str = OUTPUT_PNG_PATH  - the output path
    title: str = "Mu vs Theta — colored by TrueAction (0/1/2)"  - the title of the scatterplot
    point_size: int = 14 - the point size
    alpha: float = 0.7  - the alpha value 
    """

    fig, ax = plt.subplots(figsize=(7, 7))
    sns.scatterplot(
        data=data,
        x=x_col, y=y_col,
        hue=hue_col,
        s=point_size, alpha=alpha,
        linewidth=0, ax=ax
    )

    ## Crop
    min_lim = min(data[x_col].min(), data[y_col].min())
    max_lim = max(data[x_col].max(), data[y_col].max())

    ax.set_xlabel('Mu')
    ax.set_ylabel('Theta')
    ax.set_title(title)
    ax.set_xlim(min_lim, max_lim)
    ax.set_ylim(min_lim, max_lim)
    ax.set_aspect('equal', adjustable='box')

    ax.legend(title=hue_col, bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
    plt.tight_layout()

    fig.savefig(output_path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved scatter to: {output_path}")


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("mycsv.csv")
# Last known accuracy .405 on a single timeframe of 224 minutes.
# Heuristic .541 on M28 and now data mining commences

# Compute delta V over 3 rows (value_final - value_initial)
df["V_prev3"] = df["value"].shift(3)
df["delta_V"] = df["value"] - df["V_prev3"]

# Compare signs
df["aligned"] = np.sign(df["Margin"]) == np.sign(df["delta_V"])

# Accuracy
accuracy = df["aligned"].mean()
print("Heuristic Accuracy:", round(accuracy, 4))


FileNotFoundError: [Errno 2] No such file or directory: 'mycsv.csv'